# Tarea 3 — PyTorch

**Nombre del integrante:** Diego Goncalves

**Cédula del integrante:** 30124687

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Device configuration (GPU if available, else CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset constants
DATA_DIR = "./data"
BATCH_SIZE = 64
INPUT_SIZE = 784

# Original class names (10 classes)
ORIGINAL_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Simplified class mapping:
# 0: Top (T-shirt, Pullover, Coat, Shirt) -> [0, 2, 4, 6]
# 1: Bottom (Trouser, Dress) -> [1, 3]
# 2: Footwear (Sandal, Sneaker, Ankle boot) -> [5, 7, 9]
# 3: Bag (Bag) -> [8]
ORIGINAL_TO_SIMPLIFIED = {
    0: 0, 2: 0, 4: 0, 6: 0,  # Top
    1: 1, 3: 1,              # Bottom
    5: 2, 7: 2, 9: 2,        # Footwear
    8: 3                     # Bag
}
SIMPLIFIED_CLASSES = ["Top", "Bottom", "Footwear", "Bag"]

## 2. Datos (Fashion MNIST original y clases simplificadas)

In [ ]:
# Dataset wrapper for simplified classification
class SimplifiedFashionMNIST(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, target = self.base_dataset[idx]
        return img, ORIGINAL_TO_SIMPLIFIED[target]

# Pixel values directly in [0, 1] without external Normalize
transform = transforms.ToTensor()

# Download and load original datasets
train_dataset_orig = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=True, transform=transform, download=True
)
test_dataset_orig = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=False, transform=transform, download=True
)

# Simplified datasets
train_dataset_simp = SimplifiedFashionMNIST(train_dataset_orig)
test_dataset_simp = SimplifiedFashionMNIST(test_dataset_orig)

# DataLoaders
train_loader_orig = DataLoader(train_dataset_orig, batch_size=BATCH_SIZE, shuffle=True)
test_loader_orig = DataLoader(test_dataset_orig, batch_size=BATCH_SIZE, shuffle=False)

train_loader_simp = DataLoader(train_dataset_simp, batch_size=BATCH_SIZE, shuffle=True)
test_loader_simp = DataLoader(test_dataset_simp, batch_size=BATCH_SIZE, shuffle=False)

## 3. Modelos (rectangular y convolucional)

In [ ]:
def calculate_convoluted_layers(num_hidden, out_features=10):
    """
    Calculates layer sizes using linear interpolation:
    neurons(i) = round(784 + (out_features - 784) * i / (num_hidden + 1))
    """
    layers = []
    for i in range(1, num_hidden + 1):
        n = round(784 + (out_features - 784) * i / (num_hidden + 1))
        layers.append(n)
    return layers

def build_model(model_type="rectangular", num_hidden=2, neurons_per_layer=512, 
                classification="original", use_bn=False, dropout_rate=0.0):
    """
    Builds an nn.Sequential model meeting the PTH contract:
    nn.Flatten() -> [nn.Linear -> (nn.BatchNorm1d) -> nn.ReLU -> (nn.Dropout)] -> nn.Linear
    """
    out_features = 10 if classification == "original" else 4
    
    if model_type == "rectangular":
        hidden_sizes = [neurons_per_layer] * num_hidden
    else:  # convolutional (interpolated funnel architecture)
        hidden_sizes = calculate_convoluted_layers(num_hidden, out_features)

    layers = [nn.Flatten()]
    in_dim = INPUT_SIZE

    for h_dim in hidden_sizes:
        layers.append(nn.Linear(in_dim, h_dim))
        if use_bn:
            layers.append(nn.BatchNorm1d(h_dim))
        layers.append(nn.ReLU())
        if dropout_rate > 0.0:
            layers.append(nn.Dropout(dropout_rate))
        in_dim = h_dim

    # Output layer
    layers.append(nn.Linear(in_dim, out_features))

    return nn.Sequential(*layers)

## 4. Entrenar, probar, guardar y cargar

In [ ]:
def train_model(model, train_loader, epochs=5, lr=0.001, weight_decay=0.0):
    """Trains the network and displays the loss curve across epochs."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    model.to(DEVICE)
    loss_history = []

    print(f"\nIniciando entrenamiento por {epochs} época(s)...")
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        loss_history.append(epoch_loss)
        print(f"Época [{epoch}/{epochs}] - Pérdida (Loss): {epoch_loss:.4f}")

    # Plot loss curve
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, epochs + 1), loss_history, marker='o', color='royalblue')
    plt.title("Pérdida (Loss) por Época")
    plt.xlabel("Época")
    plt.ylabel("Pérdida")
    plt.grid(True)
    plt.show()

    return loss_history

def test_model(model, test_loader, num_classes=10):
    """Evaluates the model, showing confusion matrix, accuracy, precision, and recall."""
    model.to(DEVICE)
    model.eval()

    confusion_matrix = torch.zeros(num_classes, num_classes, dtype=torch.int64)

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            for t, p in zip(labels.view(-1), preds.view(-1)):
                confusion_matrix[t.long(), p.long()] += 1

    cm = confusion_matrix.numpy()
    
    # Accuracy
    total_samples = cm.sum()
    correct_samples = np.trace(cm)
    accuracy = correct_samples / total_samples if total_samples > 0 else 0.0

    # Macro Precision and Recall
    precisions = []
    recalls = []
    for c in range(num_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        precisions.append(prec)
        recalls.append(rec)

    precision = np.mean(precisions)
    recall = np.mean(recalls)

    print("MÉTRICAS DE RENDIMIENTO")
    print(f"Accuracy:  {accuracy * 100:.2f}%")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall:    {recall * 100:.2f}%")

    # Plot Confusion Matrix
    class_labels = ORIGINAL_CLASSES if num_classes == 10 else SIMPLIFIED_CLASSES
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title("Matriz de Confusión")
    plt.colorbar()
    tick_marks = np.arange(len(class_labels))
    plt.xticks(tick_marks, class_labels, rotation=45, ha='right')
    plt.yticks(tick_marks, class_labels)

    # Format numbers inside the matrix
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Predicción')
    plt.show()

def save_model(model, hyperparams, path):
    """Saves checkpoint following the required contract."""
    checkpoint = {
        "state_dict": model.state_dict(),
        "hyperparameters": hyperparams
    }
    torch.save(checkpoint, path)
    print(f"Modelo guardado exitosamente en '{path}'.")

def load_model(path):
    """Loads checkpoint and rebuilds nn.Sequential model."""
    if not os.path.exists(path):
        print(f"Error: El archivo '{path}' no existe.")
        return None, None

    checkpoint = torch.load(path, map_location=DEVICE)
    hp = checkpoint["hyperparameters"]

    model = build_model(
        model_type=hp.get("model_type", "rectangular"),
        num_hidden=hp.get("num_hidden", 2),
        neurons_per_layer=hp.get("neurons_per_layer", 512),
        classification=hp.get("classification", "original"),
        use_bn=hp.get("use_bn", False),
        dropout_rate=hp.get("dropout_rate", 0.0)
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.to(DEVICE)
    print(f"Modelo cargado exitosamente desde '{path}'.")
    return model, hp

## 5. Menú

La última celda debe dejar el menú corriendo (crear/cargar, entrenar, probar, guardar, salir).

In [ ]:
def menu():
    current_model = None
    current_hp = None

    while True:
        print("MENÚ PRINCIPAL - PYTORCH")
        if current_hp:
            tipo = current_hp['model_type']
            clase = current_hp['classification']
            ep = current_hp['epochs_trained']
            print(f" [Red cargada: {tipo} | {clase} | Épocas: {ep}]")
        else:
            print(" [No hay ninguna red cargada en memoria]")
        print("1. Crear una nueva red")
        print("2. Cargar una red desde archivo")
        print("3. Entrenar la red")
        print("4. Probar la red (conjunto de prueba)")
        print("5. Guardar la red en archivo *.pth")
        print("6. Salir")

        choice = input("Seleccione una opción (1-6): ").strip()

        if choice == "1":
            print("\nCreación de Red Neuronal")
            c_type = input("Tipo de clasificación [1] Original (10 clases) | [2] Simplificada (4 clases) [Por defecto: 1]: ").strip()
            classification = "simplified" if c_type == "2" else "original"

            m_type = input("Tipo de arquitectura [1] Rectangular | [2] Convolucional (embudo) [Por defecto: 1]: ").strip()
            model_type = "convolutional" if m_type == "2" else "rectangular"

            hidden_str = input("Número de capas ocultas [Por defecto: 2]: ").strip()
            num_hidden = int(hidden_str) if hidden_str.isdigit() and int(hidden_str) > 0 else 2

            neurons = 512
            if model_type == "rectangular":
                n_str = input("Neuronas por capa oculta [Por defecto: 512]: ").strip()
                neurons = int(n_str) if n_str.isdigit() and int(n_str) > 0 else 512

            # Optimizations
            opt_bn = input("¿Habilitar Batch Normalization? (s/n) [Por defecto: n]: ").strip().lower() == 's'
            drop_str = input("Dropout rate (ej. 0.2 o 0 para desactivar) [Por defecto: 0]: ").strip()
            try:
                dropout_rate = float(drop_str)
            except ValueError:
                dropout_rate = 0.0

            current_model = build_model(
                model_type=model_type,
                num_hidden=num_hidden,
                neurons_per_layer=neurons,
                classification=classification,
                use_bn=opt_bn,
                dropout_rate=dropout_rate
            )

            current_hp = {
                "model_type": model_type,
                "num_hidden": num_hidden,
                "neurons_per_layer": neurons,
                "classification": classification,
                "use_bn": opt_bn,
                "dropout_rate": dropout_rate,
                "epochs_trained": 0
            }
            print("\n¡Red creada exitosamente!")
            print(current_model)

        elif choice == "2":
            path = input("\nIngrese el nombre del archivo *.pth a cargar: ").strip()
            loaded_model, hp = load_model(path)
            if loaded_model is not None:
                current_model = loaded_model
                current_hp = hp

        elif choice == "3":
            if current_model is None:
                print("Primero debe crear o cargar una red.")
                continue

            ep_str = input("\nNúmero de épocas a entrenar [Por defecto: 5]: ").strip()
            epochs = int(ep_str) if ep_str.isdigit() and int(ep_str) > 0 else 5

            lr_str = input("Learning Rate [Por defecto: 0.001]: ").strip()
            try:
                lr = float(lr_str)
            except ValueError:
                lr = 0.001

            wd_str = input("Weight Decay (L2 regularization) [Por defecto: 0.0]: ").strip()
            try:
                wd = float(wd_str)
            except ValueError:
                wd = 0.0

            train_loader = train_loader_orig if current_hp["classification"] == "original" else train_loader_simp
            train_model(current_model, train_loader, epochs=epochs, lr=lr, weight_decay=wd)
            current_hp["epochs_trained"] = current_hp.get("epochs_trained", 0) + epochs

        elif choice == "4":
            if current_model is None:
                print("Primero debe crear o cargar una red.")
                continue

            num_classes = 10 if current_hp["classification"] == "original" else 4
            test_loader = test_loader_orig if current_hp["classification"] == "original" else test_loader_simp
            test_model(current_model, test_loader, num_classes=num_classes)

        elif choice == "5":
            if current_model is None:
                print("No hay ninguna red en memoria para guardar.")
                continue

            # Suggested default file name
            suggested = f"pytorch_fashion_1_{current_hp['classification']}_{current_hp['model_type']}.pth"
            path = input(f"\nNombre de archivo para guardar [Por defecto: {suggested}]: ").strip()
            if not path:
                path = suggested
            if not path.endswith(".pth"):
                path += ".pth"
            save_model(current_model, current_hp, path)

        elif choice == "6":
            print("\nSaliendo del programa.")
            break
        else:
            print("Opción inválida. Intente de nuevo.")

# Ejecutar el menú
menu()